<a href="https://colab.research.google.com/github/kintama285/test/blob/main/BDPAL_REYVANDI_IQBAL_M_23_11_5650.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Analisis Prediksi Harga Pangan di Indonesia Menggunakan Algoritma Linear Regression dan Decision Tree Berbasis PySpark**

#Ujian Akhir Semester<br>
# Big Data & Predictive Analytics Lanjut / 2<br>
**Nama Mahasiswa : Reyvandi Iqbal Maulana<br>
NIM : 23.11.5650<br>
Program Studi :S1 Informatika<br>
Fakultas : Ilmu Komputer<br>
Universitas Amikom Yogyakarta<br>
Tahun Akademik 2026**

Link dataset : https://www.kaggle.com/datasets/usmanlovescode/indonesia-food-prices-dataset/code

**1. Instalasi dan Inisialisasi Spark
Blok awal ini bertujuan untuk menyiapkan "mesin" pemrosesan data.**


* !pip install pyspark: Mengunduh
library Apache Spark agar bisa dijalankan di lingkungan Python.

* SparkSession.builder: Membuat pintu masuk utama untuk pemrograman Spark. Nama aplikasi UAS_BigData_AMIKOM digunakan untuk mengidentifikasi job yang sedang berjalan di cluster.

In [1]:
# Install PySpark
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Inisialisasi Spark Session [Soal No. 3]
spark = SparkSession.builder.appName("UAS_BigData_Informatika").getOrCreate()

# Load Dataset (Pasti Ada di setiap Google Colab) [Soal No. 2]
df = spark.read.csv('/content/wfp_food_prices_idn.csv', header=True, inferSchema=True)

# Menampilkan 5 data pertama
df.show(5)

+----------+----------+----------+----------------+--------+---------+-------------------+-----------+----------+----------------+----------------+---------+--------+----------+
|      date|    admin1|    admin2|          market|latitude|longitude|           category|  commodity|      unit|       priceflag|       pricetype| currency|   price|  usdprice|
+----------+----------+----------+----------------+--------+---------+-------------------+-----------+----------+----------------+----------------+---------+--------+----------+
|     #date|#adm1+name|#adm2+name|#loc+market+name|#geo+lat| #geo+lon|         #item+type| #item+name|#item+unit|#item+price+flag|#item+price+type|#currency|  #value|#value+usd|
|2007-01-15|      NULL|      NULL|National Average|    NULL|     NULL| cereals and tubers|       Rice|        KG|          actual|          Retail|      IDR| 5941.98|     0.653|
|2007-01-15|      NULL|      NULL|National Average|    NULL|     NULL| cereals and tubers|Wheat flour|        

**2. Memuat Dataset (Soal No. 1 & 2)
Langkah ini merepresentasikan penyimpanan data pada sistem file.**

* spark.read.csv(...): Fungsi untuk membaca dataset primer/sekunder Anda.

* header=True: Menginstruksikan Spark untuk menggunakan baris pertama file sebagai nama kolom.


* inferSchema=True: Spark secara otomatis menebak tipe data (misalnya angka atau teks) agar kita tidak perlu mendefinisikannya secara manual satu per satu.

In [7]:
# Casting kolom harga (price) menjadi double
df_clean = df.filter(col("price") != "#value") \
             .withColumn("price", col("price").cast("double")) \
             .na.drop(subset=["price"]) # Menghapus baris yang kosong pada kolom 'price'

print("Kualitas data terjamin: Tipe data telah sesuai dan tidak ada nilai kosong pada kolom harga.")

Kualitas data terjamin: Tipe data telah sesuai dan tidak ada nilai kosong pada kolom harga.


**3. Preprocessing dan Kualitas Data (Soal No. 3c)
Output pada bagian ini menjamin data siap untuk diolah tanpa ada gangguan nilai kosong (null).**



* withColumn("price", col("price").cast("double")): Mengubah tipe data harga dari teks menjadi angka desimal agar bisa dihitung secara matematis.


* na.drop(): Menghapus baris yang memiliki nilai kosong. Ini penting untuk menjamin Veracity (kebenaran data) dalam prinsip 5V Big Data.

In [8]:
df_clean.createOrReplaceTempView("food_prices")

# Agregasi nilai harga menggunakan Spark SQL [Soal 3d]
query_result = spark.sql("""
    SELECT admin1 as provinsi, commodity, ROUND(AVG(price), 2) as rata_rata_harga
    FROM food_prices
    GROUP BY provinsi, commodity
    ORDER BY rata_rata_harga DESC
""")
query_result.show(10)

+------------------+--------------------+---------------+
|          provinsi|           commodity|rata_rata_harga|
+------------------+--------------------+---------------+
|              ACEH|Meat (beef, first...|      144905.25|
|KALIMANTAN SELATAN|Meat (beef, first...|      144296.82|
|  KALIMANTAN BARAT|Meat (beef, first...|      143045.99|
|       DKI JAKARTA|Meat (beef, first...|      141108.41|
|KALIMANTAN SELATAN|         Meat (beef)|      140032.03|
|  KALIMANTAN BARAT|         Meat (beef)|      138406.61|
|              ACEH|         Meat (beef)|      138337.19|
|             PAPUA|Meat (beef, first...|      137989.22|
|        JAWA BARAT|Meat (beef, first...|      137273.71|
|    SUMATERA BARAT|Meat (beef, first...|       137122.4|
+------------------+--------------------+---------------+
only showing top 10 rows


**4. Analisis dengan Spark SQL (Soal No. 3d)
Bagian ini menggunakan manipulasi data tingkat lanjut melalui query SQL.**


* createOrReplaceTempView("food_prices"): Mendaftarkan DataFrame Anda sebagai tabel virtual agar bisa diproses menggunakan bahasa SQL murni.

* GROUP BY dan AVG: Menghasilkan output berupa rata-rata harga pangan per provinsi atau komoditas. Ini memberikan pandangan sistematis mengenai sebaran harga di Indonesia.

In [9]:
# Menggunakan RDD untuk menghitung sebaran data per pasar [Soal 3e]
rdd_market = df_clean.rdd.map(lambda x: (x.market, 1))
rdd_count = rdd_market.reduceByKey(lambda a, b: a + b)

print("Hasil Operasi RDD (Jumlah data per pasar):")
print(rdd_count.take(5))

Hasil Operasi RDD (Jumlah data per pasar):
[('National Average', 1663), ('Pasar Ulee Kareng', 1233), ('Pasar Kota Lhokseumawe', 1223), ('Pasar Anyar (Kab. Buleleng)', 1288), ('Pasar Kranggot', 1375)]


**5. Operasi RDD (Soal No. 3e)
Output ini menunjukkan kemampuan Anda melakukan operasi tingkat rendah (MapReduce).**

* df.rdd.map(lambda x: (x.market, 1)): Tahap Map di mana setiap baris data dipetakan menjadi pasangan (Nama Pasar, angka 1).


* reduceByKey(lambda a, b: a + b): Tahap Reduce yang menjumlahkan semua angka 1 berdasarkan kunci yang sama (Nama Pasar). Hasilnya adalah jumlah total data per lokasi pasar.

In [11]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.sql.functions import col

# Mengubah kategori menjadi angka (Indexing)
indexer = StringIndexer(inputCol="commodity", outputCol="commodity_idx")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# Cast latitude and longitude to double and drop rows with nulls in these columns
df_processed = df_indexed.withColumn("latitude", col("latitude").cast("double")) \
                         .withColumn("longitude", col("longitude").cast("double")) \
                         .na.drop(subset=["latitude", "longitude"])

# Feature Engineering
assembler = VectorAssembler(inputCols=["commodity_idx", "latitude", "longitude"], outputCol="features")
final_data = assembler.transform(df_processed).select("features", "price")
train, test = final_data.randomSplit([0.8, 0.2], seed=42)

# Model 1: Linear Regression
lr = LinearRegression(labelCol="price").fit(train)

# Model 2: Decision Tree (Untuk Tuning)
dt = DecisionTreeRegressor(labelCol="price")

**6. Permodelan Machine Learning (Soal No. 4 & 5)
Menggunakan kerangka kerja MLlib untuk melakukan prediksi masa depan.**

* StringIndexer: Mengonversi teks (seperti nama komoditas) menjadi kode angka karena algoritma mesin tidak bisa membaca teks secara langsung.


* VectorAssembler: Menggabungkan semua fitur (input) menjadi satu kolom vektor tunggal yang disebut features.


* CrossValidator & ParamGridBuilder: Melakukan Hyperparameter Tuning untuk mencari pengaturan algoritma yang paling akurat secara otomatis.

In [12]:
# Hyperparameter Tuning pada Decision Tree [Soal 5]
paramGrid = ParamGridBuilder().addGrid(dt.maxDepth, [5, 10]).build()
cv = CrossValidator(estimator=dt, estimatorParamMaps=paramGrid,
                    evaluator=RegressionEvaluator(labelCol="price"), numFolds=3)
best_model = cv.fit(train)

# Evaluasi Model [Soal 6]
predictions = best_model.transform(test)
evaluator = RegressionEvaluator(labelCol="price", metricName="rmse")
rmse = evaluator.evaluate(predictions)

print(f"Hasil Akhir Evaluasi Model (RMSE): {rmse}")

Hasil Akhir Evaluasi Model (RMSE): 9276.51525667146


**7. Evaluasi Model (Soal No. 6)
Bagian akhir untuk menentukan kesuksesan proyek.**

RegressionEvaluator: Menghitung seberapa jauh melesetnya prediksi model.


Output RMSE: Jika RMSE kecil, berarti model sangat akurat dalam memprediksi harga pangan di Indonesia berdasarkan data yang ada.